# Mega Project 4 — Delinquency Prevention
## Problem 4: POS/Cash Loan Delinquency Trajectory
## A Real DPD-Trajectory Model on POS_CASH_balance.csv (Not a Re-Derivation of MP1's SUM Features or MP3 Notebook 03's Segmentation)

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
POS/cash loans are Home Credit's other major non-revolving product,
tracked monthly with their own real days-past-due (DPD) field. A DPD
spike, or an applicant currently stuck in a real DPD streak, or repayment
progress that has stalled (remaining instalment count not shrinking as
expected) are all real leading indicators a portfolio-monitoring or
collections function wants to catch before delinquency, not after.

### Why this is genuinely new, not a repeat of MP1 or MP3 Notebook 03
`POS_CASH_balance.csv` already feeds two other real feature sets in this
suite: Mega Project 1's champion model includes real SUM totals
(application-time, not a trajectory), and Mega Project 3 Notebook 03
builds real MEAN/MAX/PCT-of-months RATE/LEVEL features for an unsupervised
segmentation. This notebook instead detects real DPD SPIKES, real DPD
STREAKS, and real repayment-PROGRESS STALLING — direction of change, not
level or rate — and trains a real SUPERVISED classifier, matching Problems
1 and 3's early-warning use case but on a third real data source. All
three siblings (MP1, MP4 Notebook 01, MP4 Notebook 03) are loaded as soft
dependencies (never retrained) purely to report real, honest side-by-side
ROC-AUC comparisons — never to claim this model replaces any of them. See
`src/features/pos_cash_trajectory_features.py`'s module docstring for the
full rationale and exact overlap disclosure.

### HYPER reuse
`src/features/pos_cash_trajectory_features.py` (new, written once for this
problem), `src/features/credit_default_features.py`,
`src/reporting/report_builder.py`, `src/utils/stats_checks.py`,
`src/utils/performance_setup.py` — reused unchanged from the rest of the
suite.

### Scope disclosure
Only applicants with at least one real prior POS/cash loan have a
trajectory signal to score — an applicant with none is out of scope, not
assigned a fabricated default. This run's real scope percentage is
reported in Section 4 below.

### Verification status (2026-09-01 policy)
Per explicit instruction, this notebook was **not** executed against any
synthetic fixture before delivery. The new feature-engineering functions
(`engineer_pos_cash_trajectory_features`, `compute_naive_current_dpd`) were
verified with small, targeted, hand-built test cases — a genuine DPD
spike, a real DPD streak, null `SK_DPD` (defensively zeroed), null
`CNT_INSTALMENT_FUTURE` (including a case where an entire real half of an
applicant's history has no valid value), a single-month applicant, and a
constant/no-DPD applicant — every one of the 8 output features checked by
hand against the input, zero nulls in any case. This file's syntax was
checked (`py_compile`/`ast.parse`, 0 errors) and this notebook passes
`nbformat.validate()`. **No champion, no AUC, no verdict has been computed
by us for this notebook.** Those are determined only by running this
notebook against your real, downloaded Home Credit dataset.


In [ ]:
# ============================================================================
# NOTEBOOK 04 — MEGA PROJECT 4: DELINQUENCY PREVENTION
# PROBLEM 4: POS/CASH LOAN DELINQUENCY TRAJECTORY — A REAL DPD-TRAJECTORY
# MODEL ON POS_CASH_balance.csv (NOT A RE-DERIVATION OF MP1's SUM FEATURES
# OR MP3 NOTEBOOK 03's SEGMENTATION OF THE SAME TABLE)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains a NEW, real supervised
# model on a feature space neither MP1's champion nor MP3 Notebook 03's
# segmentation builds: real DPD SPIKES, real DPD STREAKS, and real
# instalment-repayment-PROGRESS VELOCITY on the applicant's own previous
# Home Credit POS/cash (non-revolving instalment) loans — direction of
# change, not a static level, rate, or sum. This model, MP1's, MP4 Notebook
# 01's, and MP4 Notebook 03's all predict/relate to the same real `TARGET`
# or behavioral question, so all three are loaded as soft dependencies
# (never retrained) purely to report real, honest side-by-side ROC-AUC
# comparisons — never a claim of superiority in either direction. See
# src/features/pos_cash_trajectory_features.py's module docstring for the
# full rationale and exact overlap disclosure.
#
# WHY THIS IS THE RIGHT FOURTH PROBLEM FOR "DELINQUENCY PREVENTION": Problem
# 1 looks at instalment-LOAN payment behavior in aggregate; Problem 3 looks
# at revolving/credit-card trajectory. POS/cash loans are Home Credit's
# other major non-revolving product, tracked monthly with its own real DPD
# field — a third, independent, currently-performing-loan behavioral signal.
#
# HYPER REUSE: src/features/pos_cash_trajectory_features.py (new HYPER
# module for this problem), src/reporting/report_builder.py,
# src/utils/stats_checks.py, src/utils/performance_setup.py — all reused
# unchanged from Mega Projects 1-3 and MP4 Problems 1-3, per the suite's
# standing HYPER rule.
#
# VERIFICATION NOTE (2026-09-01 policy — see CHANGELOG): per explicit
# instruction, this notebook was NOT executed against any synthetic fixture
# before delivery. New feature logic
# (`engineer_pos_cash_trajectory_features`, `compute_naive_current_dpd`) was
# verified with small, targeted, hand-built test cases covering a genuine
# DPD spike, a real DPD streak, null SK_DPD (defensively zeroed), null
# CNT_INSTALMENT_FUTURE (including a case where an entire real half of an
# applicant's history has no valid value), a single-month applicant, and a
# constant/no-DPD applicant — every one of the 8 output features checked by
# hand against the input, zero nulls in any case. This notebook's own real
# numbers, champion, and verdict are determined ONLY by running it against
# your real, downloaded Home Credit dataset.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (identical pattern to every
# notebook in this suite).
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP1_ARTIFACTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "artifacts"
MP4_DIR = SUITE_ROOT / "04_mega_project_4_delinquency_prevention"
ARTIFACTS_DIR = MP4_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP4_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP4_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom
from utils.stats_checks import monotonic_within_noise

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import).
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2).
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, f1_score, brier_score_loss

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.pos_cash_trajectory_features import engineer_pos_cash_trajectory_features
from features.credit_default_features import engineer_credit_default_features_v2
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data (WARP: Parquet-over-CSV cache).
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR)
pos_cash = load_csv_cached(RAW_DIR / "POS_CASH_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)
print(f"[DATA] Real application_train.csv: {app.shape[0]:,} rows x {app.shape[1]} cols.")
print(f"[DATA] Real POS_CASH_balance.csv: {pos_cash.shape[0]:,} rows x {pos_cash.shape[1]} cols.")

required_cols = ["SK_ID_CURR", "TARGET"]
missing_req = [c for c in required_cols if c not in app.columns]
if missing_req:
    raise KeyError(f"Required real columns missing from application_train.csv: {missing_req}")

# ---------------------------------------------------------------------------
# SECTION 5 — HYPER feature engineering: real POS/cash DPD-trajectory
# profile, one row per SK_ID_CURR (src/features/pos_cash_trajectory_features.py).
# ---------------------------------------------------------------------------
trajectory_feat, FEATURE_COLS = engineer_pos_cash_trajectory_features(pos_cash)
print(f"[FEATURES] {trajectory_feat.height:,} real applicants have at least one real "
      f"POS_CASH_balance.csv record ({len(FEATURE_COLS)} real trajectory features engineered).")

scored = trajectory_feat.join(app.select(["SK_ID_CURR", "TARGET"]), on="SK_ID_CURR", how="inner")
N_SCOPE = scored.height
N_APP_TOTAL = app.height
print(f"[SCOPE] {N_SCOPE:,} of {N_APP_TOTAL:,} real applicants ({N_SCOPE / N_APP_TOTAL:.1%}) have real "
      f"POS/cash loan history and are in scope for this notebook -- a real, disclosed scope boundary "
      f"(an applicant with no prior POS/cash loan has no trajectory signal to score, not a fabricated "
      f"default).")

pdf = scored.to_pandas()
X_all = pdf[FEATURE_COLS].astype("float64")
y_all = pdf["TARGET"].to_numpy()
POS_RATE = float(y_all.mean())
print(f"[TARGET] Real observed default rate in scope: {POS_RATE:.4f} ({int(y_all.sum()):,} of {N_SCOPE:,}).")

# ---------------------------------------------------------------------------
# SECTION 6 — Train / holdout split (stratified, fixed seed).
# ---------------------------------------------------------------------------
X_train, X_holdout, y_train, y_holdout, id_train, id_holdout = train_test_split(
    X_all, y_all, pdf["SK_ID_CURR"].to_numpy(),
    test_size=0.25, random_state=SEED, stratify=y_all,
)
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_holdout_imp = imputer.transform(X_holdout)
print(f"[SPLIT] {len(X_train):,} train / {len(X_holdout):,} holdout real applicants "
      f"(stratified on real TARGET, SEED={SEED}).")

# ---------------------------------------------------------------------------
# SECTION 7 — 4-candidate real model screening.
# ---------------------------------------------------------------------------
def make_candidate_models():
    return {
        "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
        "decision_tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=SEED),
        "random_forest": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=10,
                                                 class_weight="balanced", random_state=SEED, n_jobs=CPU_CEILING_THREADS),
        "gradient_boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=SEED),
    }


X_screen_train, X_screen_val, y_screen_train, y_screen_val = train_test_split(
    X_train_imp, y_train, test_size=0.25, random_state=SEED, stratify=y_train,
)
screen_results = {}
for name, model in make_candidate_models().items():
    model.fit(X_screen_train, y_screen_train)
    proba = model.predict_proba(X_screen_val)[:, 1]
    auc = float(roc_auc_score(y_screen_val, proba))
    screen_results[name] = auc
    print(f"[SCREEN] {name}: real validation ROC-AUC = {auc:.4f}")

TOP2_NAMES = sorted(screen_results, key=screen_results.get, reverse=True)[:2]
print(f"[SCREEN] Top 2 real candidates advancing to 5-fold CV: {TOP2_NAMES}")

# ---------------------------------------------------------------------------
# SECTION 8 — Real 5-fold CV on the top-2 screened candidates.
# ---------------------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}
for name in TOP2_NAMES:
    fold_aucs = []
    for tr_idx, va_idx in skf.split(X_train_imp, y_train):
        model = make_candidate_models()[name]
        model.fit(X_train_imp[tr_idx], y_train[tr_idx])
        proba = model.predict_proba(X_train_imp[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train[va_idx], proba))
    cv_results[name] = {"mean_auc": float(np.mean(fold_aucs)), "std_auc": float(np.std(fold_aucs))}
    print(f"[CV] {name}: real 5-fold mean ROC-AUC = {cv_results[name]['mean_auc']:.4f} "
          f"(+/- {cv_results[name]['std_auc']:.4f})")

CHAMPION_NAME = max(cv_results, key=lambda k: cv_results[k]["mean_auc"])
RUNNER_UP_NAME = [n for n in TOP2_NAMES if n != CHAMPION_NAME][0]
print(f"[CHAMPION] {CHAMPION_NAME} selected by highest real mean 5-fold CV ROC-AUC among the "
      f"top-2 screened candidates.")

# ---------------------------------------------------------------------------
# SECTION 9 — Retrain champion on full train, evaluate on true holdout.
# ---------------------------------------------------------------------------
champion = make_candidate_models()[CHAMPION_NAME]
champion.fit(X_train_imp, y_train)
holdout_proba = champion.predict_proba(X_holdout_imp)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)
metrics = {
    "roc_auc": float(roc_auc_score(y_holdout, holdout_proba)),
    "precision": float(precision_score(y_holdout, holdout_pred, zero_division=0)),
    "recall": float(recall_score(y_holdout, holdout_pred, zero_division=0)),
    "f1": float(f1_score(y_holdout, holdout_pred, zero_division=0)),
    "brier_score": float(brier_score_loss(y_holdout, holdout_proba)),
}
print(f"[HOLDOUT] {CHAMPION_NAME} real holdout metrics: {json.dumps(metrics, indent=2)}")

# ---------------------------------------------------------------------------
# SECTION 10 — Real bootstrap 95% CI on holdout ROC-AUC.
# ---------------------------------------------------------------------------
N_BOOTSTRAP = 500
boot_aucs = []
_holdout_idx = np.arange(len(y_holdout))
for _ in range(N_BOOTSTRAP):
    idx = rng.choice(_holdout_idx, size=len(_holdout_idx), replace=True)
    y_bs = y_holdout[idx]
    if len(np.unique(y_bs)) < 2:
        continue
    boot_aucs.append(roc_auc_score(y_bs, holdout_proba[idx]))
boot_aucs = np.array(boot_aucs)
AUC_CI_LOW, AUC_CI_HIGH = float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))
print(f"[VALIDATION] Real {len(boot_aucs)}-resample bootstrap 95% CI on holdout ROC-AUC: "
      f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]")

# ---------------------------------------------------------------------------
# SECTION 11 — Real decile calibration check.
# ---------------------------------------------------------------------------
decile_df = pd.DataFrame({"proba": holdout_proba, "target": y_holdout})
decile_df["decile"] = pd.qcut(decile_df["proba"], q=10, labels=False, duplicates="drop")
decile_agg = (
    decile_df.groupby("decile")
    .agg(n=("target", "size"), mean_proba=("proba", "mean"), real_default_rate=("target", "mean"))
    .reset_index()
    .sort_values("decile", ascending=False)
)
rates_highest_first = decile_agg["real_default_rate"].tolist()
counts_highest_first = decile_agg["n"].tolist()
is_monotonic, _monotonicity_detail = monotonic_within_noise(rates_highest_first, counts_highest_first, alpha=0.05)
print(f"[VALIDATION] Real decile calibration monotonicity (statistically-tolerant "
      f"Bonferroni-corrected check): {'HOLDS' if is_monotonic else 'DOES NOT HOLD'}")
for _, row in decile_agg.iterrows():
    print(f"[DECILE] decile={int(row['decile'])}: n={int(row['n']):,}, mean predicted risk="
          f"{row['mean_proba']:.4f}, real default rate={row['real_default_rate']:.4f}")

# ---------------------------------------------------------------------------
# SECTION 12 — SHAP explainability (champion model only).
# ---------------------------------------------------------------------------
SHAP_SAMPLE_N = min(300, len(X_holdout_imp))
_shap_sample_idx = rng.choice(len(X_holdout_imp), size=SHAP_SAMPLE_N, replace=False)
X_shap_sample = X_holdout_imp[_shap_sample_idx]
try:
    if CHAMPION_NAME in ("decision_tree", "random_forest", "gradient_boosting"):
        shap_explainer = shap.TreeExplainer(champion)
        _raw_shap = shap_explainer.shap_values(X_shap_sample)
    else:
        _bg = shap.sample(X_train_imp, min(100, len(X_train_imp)), random_state=SEED)
        shap_explainer = shap.LinearExplainer(champion, _bg)
        _raw_shap = shap_explainer.shap_values(X_shap_sample)
    if isinstance(_raw_shap, list):
        SHAP_VALUES = np.array(_raw_shap[1]) if len(_raw_shap) > 1 else np.array(_raw_shap[0])
    elif isinstance(_raw_shap, np.ndarray) and _raw_shap.ndim == 3:
        SHAP_VALUES = _raw_shap[:, :, 1]
    else:
        SHAP_VALUES = np.array(_raw_shap)
    SHAP_MEAN_ABS = np.abs(SHAP_VALUES).mean(axis=0)
    SHAP_OK = True
except Exception as e:
    print(f"[SHAP] Real SHAP computation failed for champion {CHAMPION_NAME}: {e} -- "
          f"proceeding without explainability rather than fabricating importances.")
    SHAP_MEAN_ABS = np.zeros(len(FEATURE_COLS))
    SHAP_OK = False
shap_rank = sorted(zip(FEATURE_COLS, SHAP_MEAN_ABS), key=lambda t: t[1], reverse=True)
if SHAP_OK:
    print(f"[SHAP] Real SHAP explainability computed for champion {CHAMPION_NAME} on a real "
          f"{SHAP_SAMPLE_N}-row sample of the holdout set. Top real driver: {shap_rank[0][0]}.")

# ---------------------------------------------------------------------------
# SECTION 13a — SOFT DEPENDENCY 1: MP1 Notebook 01's real champion.
# ---------------------------------------------------------------------------
UPSTREAM_MODEL_PATH = MP1_ARTIFACTS_DIR / "notebook_01_champion_model.joblib"
MP1_COMPARISON_AVAILABLE = UPSTREAM_MODEL_PATH.exists()
MP1_COMPARISON_AUC = None
MP1_CHAMPION_NAME = None
if MP1_COMPARISON_AVAILABLE:
    try:
        bundle = joblib.load(UPSTREAM_MODEL_PATH)
        up_model = bundle["model"]
        up_ord_enc = bundle["ordinal_encoder"]
        up_imputer = bundle["imputer"]
        up_feature_cols = bundle["feature_cols"]
        up_numeric = bundle["numeric_features"]
        up_categorical = bundle["categorical_features"]
        MP1_CHAMPION_NAME = bundle["champion_name"]

        bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR)
        bureau_balance = load_csv_cached(RAW_DIR / "bureau_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
        previous_application = load_csv_cached(RAW_DIR / "previous_application.csv", PARQUET_CACHE_DIR,
                                                null_values=["", "NA", "XNA", "XAP"])
        installments = load_csv_cached(RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
        credit_card = load_csv_cached(RAW_DIR / "credit_card_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
        mp1_feat_df, mp1_numeric, mp1_categorical = engineer_credit_default_features_v2(
            app, bureau, bureau_balance, previous_application, pos_cash, installments, credit_card
        )
        if mp1_numeric != up_numeric or mp1_categorical != up_categorical:
            raise ValueError("MP1's trained feature set no longer matches the current "
                              "credit_default_features_v2 output -- skipping the comparison.")
        _mp1_pdf = mp1_feat_df.to_pandas()
        _mp1_pdf = _mp1_pdf[_mp1_pdf["SK_ID_CURR"].isin(id_holdout)].set_index("SK_ID_CURR").loc[id_holdout].reset_index()
        _mX = _mp1_pdf[up_feature_cols].copy()
        for c in up_categorical:
            _mX[c] = _mX[c].astype(object).fillna("Missing").astype(str)
        if up_categorical:
            _mX[up_categorical] = up_ord_enc.transform(_mX[up_categorical].astype(str))
        _mX[up_numeric] = up_imputer.transform(_mX[up_numeric])
        mp1_holdout_proba = up_model.predict_proba(_mX)[:, 1]
        MP1_COMPARISON_AUC = float(roc_auc_score(y_holdout, mp1_holdout_proba))
        print(f"[COMPARISON] Real MP1 Notebook 01 champion ({MP1_CHAMPION_NAME}, application-time "
              f"features) scores {MP1_COMPARISON_AUC:.4f} ROC-AUC on this notebook's SAME "
              f"{len(y_holdout):,}-row real holdout population, vs. this notebook's "
              f"{metrics['roc_auc']:.4f} using only POS/cash DPD-trajectory features.")
    except Exception as e:
        print(f"[COMPARISON] Real MP1 comparison could not be completed ({e}) -- this is a SOFT "
              f"dependency, so this notebook's own result is still complete and standalone.")
        MP1_COMPARISON_AVAILABLE = False
else:
    print("[COMPARISON] MP1 Notebook 01's real champion model was not found -- this is a SOFT "
          "dependency. Run Mega Project 1 / Notebook 01 first for the real side-by-side comparison.")


def _compare_against_sibling_scores(csv_path: Path, score_col: str, label: str):
    """Real, honest AUC comparison of an already-saved real per-applicant
    score against this notebook's holdout, on the real intersecting subset
    of applicants. Soft dependency -- returns (available, auc, n_overlap)."""
    if not csv_path.exists():
        print(f"[COMPARISON] {label}'s real per-applicant scores were not found -- this is a SOFT "
              f"dependency. Run {label} first for the real side-by-side comparison.")
        return False, None, 0
    try:
        sib_scores = pd.read_csv(csv_path)[["SK_ID_CURR", score_col]]
        holdout_df = pd.DataFrame({"SK_ID_CURR": id_holdout, "TARGET": y_holdout})
        joined = holdout_df.merge(sib_scores, on="SK_ID_CURR", how="inner")
        n_overlap = len(joined)
        if n_overlap >= 20 and joined["TARGET"].nunique() == 2:
            auc = float(roc_auc_score(joined["TARGET"], joined[score_col]))
            print(f"[COMPARISON] Real {label} score scores {auc:.4f} ROC-AUC on the {n_overlap:,} "
                  f"real applicants this notebook's holdout shares with {label}'s own scope -- two "
                  f"real, complementary post-approval behavioral signals, not competing claims.")
            return True, auc, n_overlap
        print(f"[COMPARISON] Only {n_overlap:,} real applicants overlap between this notebook's "
              f"holdout and {label}'s scope (or the overlap has only one real TARGET class) -- too "
              f"few for a meaningful real AUC comparison; skipping.")
        return False, None, n_overlap
    except Exception as e:
        print(f"[COMPARISON] Real {label} comparison could not be completed ({e}) -- this is a SOFT "
              f"dependency, so this notebook's own result is still complete and standalone.")
        return False, None, 0


# ---------------------------------------------------------------------------
# SECTION 13b — SOFT DEPENDENCY 2: MP4 Notebook 01 (installment behavior).
# ---------------------------------------------------------------------------
NB01_COMPARISON_AVAILABLE, NB01_COMPARISON_AUC, NB01_COMPARISON_N = _compare_against_sibling_scores(
    ARTIFACTS_DIR / "notebook_01_delinquency_scores.csv", "EARLY_DELINQUENCY_RISK_SCORE", "MP4 Notebook 01"
)

# ---------------------------------------------------------------------------
# SECTION 13c — SOFT DEPENDENCY 3: MP4 Notebook 03 (revolving distress).
# ---------------------------------------------------------------------------
NB03_COMPARISON_AVAILABLE, NB03_COMPARISON_AUC, NB03_COMPARISON_N = _compare_against_sibling_scores(
    ARTIFACTS_DIR / "notebook_03_distress_scores.csv", "REVOLVING_DISTRESS_RISK_SCORE", "MP4 Notebook 03"
)

# ---------------------------------------------------------------------------
# SECTION 14 — STATISTICAL ROBUSTNESS VERDICT.
# ---------------------------------------------------------------------------
validation_checks = [
    ("champion_auc_above_random", metrics["roc_auc"] > 0.5),
    ("champion_in_screened_top2", CHAMPION_NAME in TOP2_NAMES),
    ("holdout_auc_ci_excludes_random", AUC_CI_LOW > 0.5),
    ("decile_calibration_monotonic", is_monotonic),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "STATISTICALLY ROBUST — RECOMMENDED FOR PRODUCTION" if ANALYSIS_ROBUST
    else "NOT YET STATISTICALLY ROBUST — failed: " + ", ".join(_failed_validation_checks)
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Statistical robustness verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 15 — Inline charts (vivid multicolor).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

fpr, tpr, _ = roc_curve(y_holdout, holdout_proba)
axes[0].plot(fpr, tpr, label=f"{CHAMPION_NAME} (AUC={metrics['roc_auc']:.4f})", color=VIVID_PALETTE[1], linewidth=2.5)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="#999999")
axes[0].set_title("ROC Curve — Champion (Real Holdout)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

_top10 = shap_rank[:10][::-1]
axes[1].barh([t[0] for t in _top10], [t[1] for t in _top10], color=_palette(len(_top10)))
axes[1].set_title(f"SHAP Feature Importance — Champion ({CHAMPION_NAME})")
axes[1].set_xlabel("Mean |SHAP value|")

axes[2].bar(decile_agg["decile"].astype(str), decile_agg["real_default_rate"], color=VIVID_PALETTE[2])
axes[2].set_title("Real Default Rate by Predicted-Risk Decile (10 = highest risk)")
axes[2].set_xlabel("Decile")
axes[2].set_ylabel("Real observed default rate")

plt.tight_layout()
CHART_PATH = REPORTS_DIR / "notebook_04_charts.png"
plt.savefig(CHART_PATH, dpi=130, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# SECTION 16 — Save real trained artifacts.
# ---------------------------------------------------------------------------
MODEL_PATH = ARTIFACTS_DIR / "notebook_04_champion_model.joblib"
joblib.dump({
    "model": champion,
    "imputer": imputer,
    "feature_cols": FEATURE_COLS,
    "champion_name": CHAMPION_NAME,
    "cv_results": cv_results,
    "holdout_metrics": metrics,
}, MODEL_PATH)
print(f"[SAVE] Real trained champion bundle saved: {MODEL_PATH}")

full_proba = champion.predict_proba(imputer.transform(X_all))[:, 1]
scores_df = pd.DataFrame({
    "SK_ID_CURR": pdf["SK_ID_CURR"].to_numpy(),
    "POS_CASH_TRAJECTORY_RISK_SCORE": full_proba,
    "TARGET": y_all,
})
scores_path = ARTIFACTS_DIR / "notebook_04_trajectory_scores.csv"
scores_df.to_csv(scores_path, index=False)
print(f"[SAVE] Real per-applicant POS/cash-trajectory-risk scores saved: {scores_path}")

summary_artifact = {
    "notebook": "04_pos_cash_delinquency_trajectory",
    "mega_project": 4,
    "n_scope": N_SCOPE,
    "n_app_total": N_APP_TOTAL,
    "real_default_rate": POS_RATE,
    "champion_model": CHAMPION_NAME,
    "runner_up_model": RUNNER_UP_NAME,
    "cv_results": cv_results,
    "holdout_metrics": metrics,
    "holdout_auc_ci": [AUC_CI_LOW, AUC_CI_HIGH],
    "decile_calibration_monotonic": bool(is_monotonic),
    "top_shap_driver": shap_rank[0][0] if SHAP_OK else None,
    "mp1_comparison_available": MP1_COMPARISON_AVAILABLE,
    "mp1_champion_name": MP1_CHAMPION_NAME,
    "mp1_comparison_auc": MP1_COMPARISON_AUC,
    "nb01_comparison_available": NB01_COMPARISON_AVAILABLE,
    "nb01_comparison_auc": NB01_COMPARISON_AUC,
    "nb01_comparison_n": NB01_COMPARISON_N,
    "nb03_comparison_available": NB03_COMPARISON_AVAILABLE,
    "nb03_comparison_auc": NB03_COMPARISON_AUC,
    "nb03_comparison_n": NB03_COMPARISON_N,
    "analysis_verdict": ANALYSIS_VERDICT,
    "analysis_robust": ANALYSIS_ROBUST,
}
summary_path = REPORTS_DIR / "notebook_04_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_artifact, f, indent=2)
print(f"[SAVE] Real summary artifact saved: {summary_path}")

# ---------------------------------------------------------------------------
# SECTION 17 — Real Pipeline Integrity Checks.
# ---------------------------------------------------------------------------
integrity_checks = [
    ("model_bundle_saved", MODEL_PATH.exists()),
    ("scores_csv_saved", scores_path.exists()),
    ("summary_json_saved", summary_path.exists()),
    ("no_nan_in_holdout_proba", not np.isnan(holdout_proba).any()),
    ("scores_in_valid_range", bool((full_proba >= 0).all() and (full_proba <= 1).all())),
    ("feature_cols_match_bundle", FEATURE_COLS == joblib.load(MODEL_PATH)["feature_cols"]),
]
INTEGRITY_OK = all(ok for _, ok in integrity_checks)
for name, ok in integrity_checks:
    print(f"[INTEGRITY-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[INTEGRITY] Pipeline structural integrity: {'PASS' if INTEGRITY_OK else 'FAIL'}")

# ---------------------------------------------------------------------------
# SECTION 18 — Reports (HTML dashboard + Word + Excel).
# ---------------------------------------------------------------------------
DEPLOY_STATUS_WORD = "meets" if ANALYSIS_ROBUST else "does not yet meet"
DEPLOYMENT_VERDICT = ANALYSIS_VERDICT

exec_summary = [
    f"{CHAMPION_NAME} selected as champion by highest real mean 5-fold CV ROC-AUC "
    f"({cv_results[CHAMPION_NAME]['mean_auc']:.4f}) among the top-2 screened real candidates.",
    f"Real holdout ROC-AUC: {metrics['roc_auc']:.4f} (95% bootstrap CI "
    f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]) on {len(y_holdout):,} real applicants.",
    f"Real decile calibration monotonicity: {'HOLDS' if is_monotonic else 'DOES NOT HOLD'}.",
    f"Scope: {N_SCOPE:,} of {N_APP_TOTAL:,} real applicants ({N_SCOPE / N_APP_TOTAL:.1%}) with "
    f"real prior POS/cash loan history.",
]
if MP1_COMPARISON_AVAILABLE:
    exec_summary.append(f"Real comparison vs. MP1 Notebook 01: {MP1_COMPARISON_AUC:.4f} ROC-AUC "
                         f"(application-time), vs. this notebook's {metrics['roc_auc']:.4f}.")
if NB01_COMPARISON_AVAILABLE:
    exec_summary.append(f"Real comparison vs. MP4 Notebook 01: {NB01_COMPARISON_AUC:.4f} ROC-AUC "
                         f"on the {NB01_COMPARISON_N:,}-row real overlap.")
if NB03_COMPARISON_AVAILABLE:
    exec_summary.append(f"Real comparison vs. MP4 Notebook 03: {NB03_COMPARISON_AUC:.4f} ROC-AUC "
                         f"on the {NB03_COMPARISON_N:,}-row real overlap.")
exec_summary.append(f"Deployment verdict: this analysis {DEPLOY_STATUS_WORD} the deployment-readiness bar.")

sections = [
    {"heading": "Real Model Screening & Champion Selection",
     "paragraphs": [f"4 real candidate models were screened on a held-out validation split; the "
                    f"top 2 ({', '.join(TOP2_NAMES)}) advanced to real 5-fold cross-validation."],
     "table": {"headers": ["Model", "Screen ROC-AUC", "5-Fold CV Mean AUC", "5-Fold CV Std"],
               "rows": [[name, f"{screen_results[name]:.4f}",
                         f"{cv_results[name]['mean_auc']:.4f}" if name in cv_results else "n/a",
                         f"{cv_results[name]['std_auc']:.4f}" if name in cv_results else "n/a"]
                        for name in screen_results]}},
    {"heading": "Real Holdout Performance",
     "paragraphs": [f"Champion ({CHAMPION_NAME}) evaluated on a true, held-out set of "
                    f"{len(y_holdout):,} real applicants never seen during training or model selection."],
     "table": {"headers": ["Metric", "Value"],
               "rows": [[k, f"{v:.4f}"] for k, v in metrics.items()]},
     "image_path": CHART_PATH},
    {"heading": "Real Decile Calibration",
     "paragraphs": [f"Real observed default rate by predicted-risk decile (10 = highest predicted "
                    f"risk). Monotonicity check: {'HOLDS' if is_monotonic else 'DOES NOT HOLD'}."],
     "table": {"headers": ["Decile", "N", "Mean Predicted Risk", "Real Default Rate"],
               "rows": [[int(r["decile"]), int(r["n"]), f"{r['mean_proba']:.4f}", f"{r['real_default_rate']:.4f}"]
                        for _, r in decile_agg.iterrows()]}},
    {"heading": f"SHAP Explainability (Champion Model: {CHAMPION_NAME})",
     "paragraphs": [f"Real SHAP values computed on a real {SHAP_SAMPLE_N}-row sample of the holdout "
                    f"set." if SHAP_OK else "SHAP computation was not available for this run."],
     "table": {"headers": ["Feature", "Mean |SHAP value|"],
               "rows": [[n, f"{v:.5f}"] for n, v in shap_rank[:10]]}},
]
_comparison_rows = [[f"MP4 NB04 — {CHAMPION_NAME}", "POS/cash DPD trajectory", f"{metrics['roc_auc']:.4f}"]]
if MP1_COMPARISON_AVAILABLE:
    _comparison_rows.append([f"MP1 NB01 — {MP1_CHAMPION_NAME}", "Application-time covariates", f"{MP1_COMPARISON_AUC:.4f}"])
if NB01_COMPARISON_AVAILABLE:
    _comparison_rows.append([f"MP4 NB01 (on {NB01_COMPARISON_N:,}-row overlap)", "Installment-payment behavior", f"{NB01_COMPARISON_AUC:.4f}"])
if NB03_COMPARISON_AVAILABLE:
    _comparison_rows.append([f"MP4 NB03 (on {NB03_COMPARISON_N:,}-row overlap)", "Revolving/credit-card distress", f"{NB03_COMPARISON_AUC:.4f}"])
if len(_comparison_rows) > 1:
    sections.append({
        "heading": "Real Comparison vs. Other Real Champion Models",
        "paragraphs": [f"Real, honest AUC comparisons on the applicable real holdout/overlap "
                       f"population(s) -- not a claim that any model replaces another."],
        "table": {"headers": ["Model", "Data Source", "Real ROC-AUC"], "rows": _comparison_rows},
    })

insights = [{
    "headline": f"{CHAMPION_NAME} {DEPLOY_STATUS_WORD} the deployment-readiness bar",
    "specific": f"Holdout ROC-AUC {metrics['roc_auc']:.4f} (95% CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]) "
                f"on {len(y_holdout):,} real applicants using only real POS/cash DPD-trajectory features.",
    "measurable": f"Real decile calibration monotonicity {'holds' if is_monotonic else 'does not hold'}; "
                  f"top real driver: {shap_rank[0][0] if SHAP_OK else 'n/a'}.",
    "achievable": "Score is computable for any applicant with at least one real prior POS/cash loan -- "
                  f"{N_SCOPE / N_APP_TOTAL:.1%} of this population.",
    "relevant": "Trajectory signal (DPD spikes/streaks, progress velocity) is available only "
                "post-approval, complementing MP1, MP4 Notebook 01, and MP4 Notebook 03 for ongoing "
                "portfolio monitoring.",
    "timebound": "Re-score on a rolling basis as new POS_CASH_balance records accrue; re-validate "
                 "calibration each time the champion is retrained.",
}]

build_word_report(
    REPORTS_DIR / "notebook_04_report.docx",
    title="Mega Project 4 — Notebook 04: POS/Cash Loan Delinquency Trajectory",
    subtitle=f"Champion: {CHAMPION_NAME} | Real holdout ROC-AUC: {metrics['roc_auc']:.4f} | {DEPLOYMENT_VERDICT}",
    exec_summary=exec_summary, sections=sections, insights=insights,
)
print("[REPORT] Real Word report written.")

assumptions = {"random_seed": SEED, "test_size": 0.25, "n_bootstrap": N_BOOTSTRAP, "champion_model": CHAMPION_NAME}
assumption_notes = {
    "random_seed": "Fixed seed for reproducibility across every train/test split and model fit.",
    "test_size": "Fraction of in-scope applicants held out for real, unbiased evaluation.",
    "n_bootstrap": "Real resamples used for the holdout ROC-AUC 95% confidence interval.",
    "champion_model": "Selected by highest real mean 5-fold CV ROC-AUC among the top-2 screened candidates.",
}
write_csv_outputs({
    "model_screening": pd.DataFrame([{"model": k, "screen_auc": v} for k, v in screen_results.items()]),
    "decile_calibration": decile_agg,
    "shap_importance": pd.DataFrame(shap_rank, columns=["feature", "mean_abs_shap"]),
}, REPORTS_DIR)
_screening_rows = [[k, round(v, 6), round(cv_results.get(k, {}).get("mean_auc", float("nan")), 6) if k in cv_results else None]
                   for k, v in screen_results.items()]
_decile_rows = [[int(r["decile"]), int(r["n"]), round(float(r["mean_proba"]), 6), round(float(r["real_default_rate"]), 6)]
                for _, r in decile_agg.iterrows()]
_shap_rows = [[n, round(float(v), 6)] for n, v in shap_rank]
_holdout_rows = [[k, round(v, 6)] for k, v in metrics.items()]
build_excel_workbook(
    REPORTS_DIR / "notebook_04_workbook.xlsx",
    assumptions=assumptions, assumption_notes=assumption_notes,
    data_sheets=[
        {"name": "Model Screening", "headers": ["model", "screen_auc", "cv_mean_auc"], "rows": _screening_rows,
         "highlight_col": "screen_auc"},
        {"name": "Decile Calibration", "headers": ["decile", "n", "mean_proba", "real_default_rate"],
         "rows": _decile_rows, "highlight_col": "real_default_rate"},
        {"name": "SHAP Importance", "headers": ["feature", "mean_abs_shap"], "rows": _shap_rows,
         "highlight_col": "mean_abs_shap"},
        {"name": "Holdout Metrics", "headers": ["metric", "value"], "rows": _holdout_rows},
    ],
)
print("[REPORT] Real Excel workbook written.")

build_html_dashboard(
    REPORTS_DIR / "notebook_04_dashboard.html",
    title="Mega Project 4 — POS/Cash Loan Delinquency Trajectory",
    subtitle=f"Champion: {CHAMPION_NAME} | Real holdout ROC-AUC: {metrics['roc_auc']:.4f} | {DEPLOYMENT_VERDICT}",
    kpi_cards=[
        {"label": "Champion Model", "value": CHAMPION_NAME},
        {"label": "Holdout ROC-AUC", "value": f"{metrics['roc_auc']:.4f}"},
        {"label": "Real Applicants In Scope", "value": f"{N_SCOPE:,}"},
        {"label": "Real Default Rate", "value": f"{POS_RATE:.2%}"},
    ],
    charts=[
        {"id": "screenChart", "title": "Real Model Screening — Validation ROC-AUC", "type": "bar",
         "labels": list(screen_results.keys()),
         "datasets": [{"label": "Validation ROC-AUC", "data": list(screen_results.values()),
                       "backgroundColor": [VIVID_PALETTE[1] if n == CHAMPION_NAME else VIVID_PALETTE[4]
                                           for n in screen_results]}]},
        {"id": "decileChart", "title": "Real Default Rate by Predicted-Risk Decile", "type": "bar",
         "labels": [str(int(d)) for d in decile_agg["decile"]],
         "datasets": [{"label": "Real default rate", "data": decile_agg["real_default_rate"].tolist(),
                       "backgroundColor": VIVID_PALETTE[2]}]},
        {"id": "shapChart", "title": f"SHAP Feature Importance — Champion ({CHAMPION_NAME})", "type": "bar",
         "labels": [t[0] for t in shap_rank[:10]],
         "datasets": [{"label": "Mean |SHAP value|", "data": [float(t[1]) for t in shap_rank[:10]],
                       "backgroundColor": VIVID_PALETTE[3]}]},
    ],
    insights=insights,
    data_table={"title": "Real Per-Applicant POS/Cash-Trajectory-Risk Scores (sample)",
                "columns": ["SK_ID_CURR", "POS_CASH_TRAJECTORY_RISK_SCORE", "TARGET"],
                "rows": scores_df.head(300).values.tolist()},
)
print("[REPORT] Real HTML dashboard written.")

print(f"\n[DONE] Notebook 04 complete in {time.time() - T0:.1f}s. "
      f"Champion: {CHAMPION_NAME}. Holdout ROC-AUC: {metrics['roc_auc']:.4f}. "
      f"Verdict: {ANALYSIS_VERDICT}.")
